# 🔬 Final Phase — Notebook 3: Quantum Interpretability Visualizations
## Understanding WHY the Quantum Layer Helps

This notebook generates research-quality figures showing:
1. **Qubit Activation Heatmaps** — which qubits activate for which license plate regions
2. **Quantum Feature Entanglement Map** — cross-qubit correlations the classical model can't capture
3. **Visually Similar Character Analysis** — how quantum distinguishes '0' vs 'O', '1' vs 'I', '5' vs 'S'
4. **ZeroDCE Enhancement Quality** — enhancement effectiveness across lighting conditions
5. **Full Training Curve Comparison** — from saved history files

## Cell 1 — Setup

In [ ]:
!pip install pennylane pennylane-lightning editdistance seaborn -q

import os, json
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from torchvision import transforms
from PIL import Image
import pandas as pd
import pennylane as qml
from torch.utils.data import Dataset, random_split
import editdistance

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Device: {device}')

from google.colab import drive
drive.mount('/content/drive')

## Cell 2 — Config, Models, Dataset (shared setup)

In [ ]:
PROJECT_PATH   = '/content/drive/MyDrive/Quantum_LPR_Project'
CHECKPOINT_DIR = os.path.join(PROJECT_PATH, 'checkpoints')
CSV_PATH       = '/content/drive/MyDrive/License Plate/Manifests/2_train_hr_images.csv'
EXTRACT_PATH   = '/content/lpr_train'
SEED           = 42

CHARS    = '0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ'
CHAR2IDX = {c: i+1 for i, c in enumerate(CHARS)}
IDX2CHAR = {i+1: c for i, c in enumerate(CHARS)}
N_QUBITS = 8

# ── Model definitions ─────────────────────────────────────
class ZeroDCE_Light(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3,16,3,1,1), nn.ReLU(),
            nn.Conv2d(16,16,3,1,1), nn.ReLU(),
            nn.Conv2d(16,24,3,1,1), nn.Tanh())
    def forward(self, x):
        A, e = self.conv(x), x
        for i in range(8): e = e + A[:,3*i:3*(i+1)] * (torch.pow(e,2) - e)
        return e

dev_qml = qml.device('default.qubit', wires=N_QUBITS)

@qml.qnode(dev_qml, interface='torch')
def quantum_circuit(inputs, weights):
    qml.templates.AngleEmbedding(inputs, wires=range(N_QUBITS))
    qml.templates.StronglyEntanglingLayers(weights, wires=range(N_QUBITS))
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]

class QuantumLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.q_layer = qml.qnn.TorchLayer(quantum_circuit,
                                           {'weights': (2, N_QUBITS, 3)})
    def forward(self, x): return self.q_layer(x)

class HybridLPRNet_8Q(nn.Module):
    def __init__(self, num_classes=37):
        super().__init__()
        self.enhancer  = ZeroDCE_Light()
        self.features  = nn.Sequential(
            nn.Conv2d(3,64,3,1,1), nn.MaxPool2d(2), nn.ReLU(),
            nn.Conv2d(64,128,3,1,1), nn.MaxPool2d(2), nn.ReLU(),
            nn.Conv2d(128,N_QUBITS,1,1))
        self.quantum   = QuantumLayer()
        self.rnn       = nn.LSTM(N_QUBITS, 128, bidirectional=True, batch_first=True)
        self.classifier= nn.Linear(256, num_classes)
    def forward(self, x):
        x = self.enhancer(x)
        x = self.features(x)
        b,c,h,w = x.size()
        x_seq = x.mean(dim=2).permute(0,2,1)          # [B, W, 8]
        q_out = self.quantum(x_seq.reshape(-1,N_QUBITS)).reshape(b,w,N_QUBITS)
        return self.classifier(self.rnn(q_out)[0]).permute(1,0,2)

    def get_pre_quantum(self, x):
        """Returns [B, W, 8] features going INTO the quantum layer."""
        with torch.no_grad():
            x = self.enhancer(x)
            x = self.features(x)
            return x.mean(dim=2).permute(0,2,1)       # [B, W, 8]

    def get_post_quantum(self, x):
        """Returns [B, W, 8] features coming OUT of the quantum layer."""
        with torch.no_grad():
            pre = self.get_pre_quantum(x)
            b, w, _ = pre.shape
            return self.quantum(pre.reshape(-1,N_QUBITS)).reshape(b,w,N_QUBITS)

# ── Load model ────────────────────────────────────────────
q_model = HybridLPRNet_8Q(37).to(device)

for fname in ['8qubit_best.pth', '8qubit_model.pth']:
    path = os.path.join(CHECKPOINT_DIR, fname)
    if os.path.exists(path):
        ckpt = torch.load(path, map_location=device)
        q_model.load_state_dict(ckpt['model_state_dict'])
        print(f'✅ Loaded {fname}')
        break

q_model.eval()

# ── Dataset ───────────────────────────────────────────────
transform = transforms.Compose([transforms.Resize((64,256)), transforms.ToTensor()])

class LPRDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None, mode='eval'):
        self.data_frame = pd.read_csv(csv_file)
        self.root_dir   = root_dir
        self.transform  = transform
        self.mode       = mode
    def __len__(self): return len(self.data_frame)
    def __getitem__(self, idx):
        row      = self.data_frame.iloc[idx]
        img_path = row['path']
        label    = str(row['label']).upper()
        full_path = img_path if img_path.startswith('/content') \
                    else os.path.join(self.root_dir, img_path)
        try:   img = Image.open(full_path).convert('RGB')
        except: img = Image.new('RGB', (256, 64))
        if self.transform: img = self.transform(img)
        return img, label

dataset = LPRDataset(CSV_PATH, EXTRACT_PATH, transform=transform)
print(f'✅ Dataset: {len(dataset)} images')

## Cell 3 — Visualization 1: Qubit Activation Heatmap

In [ ]:
def plot_qubit_heatmap(model, img_tensor, label, title_prefix=''):
    """
    Show:
    - Row 1: Pre-quantum CNN features (8 channels × Width)
    - Row 2: Post-quantum transformed features (8 channels × Width)
    This visualizes what the quantum transformation does.
    """
    inp  = img_tensor.unsqueeze(0).to(device)
    pre  = model.get_pre_quantum(inp).cpu().squeeze(0).numpy()   # [W, 8]
    post = model.get_post_quantum(inp).cpu().squeeze(0).numpy()  # [W, 8]

    fig, axes = plt.subplots(2, 1, figsize=(16, 5))
    fig.suptitle(f'{title_prefix}  Label: {label}',
                 fontsize=13, fontweight='bold')

    im0 = axes[0].imshow(pre.T, aspect='auto', cmap='viridis')
    axes[0].set_title('CNN Features → Quantum Input  (pre-quantum)', fontsize=11)
    axes[0].set_ylabel('Qubit Index (0–7)')
    axes[0].set_xlabel('Image Width (scanning left → right)')
    axes[0].set_yticks(range(N_QUBITS))
    plt.colorbar(im0, ax=axes[0])

    im1 = axes[1].imshow(post.T, aspect='auto', cmap='magma')
    axes[1].set_title('Quantum-Transformed Features  (post-quantum)',
                      fontsize=11)
    axes[1].set_ylabel('Qubit Index (0–7)')
    axes[1].set_xlabel('Image Width (scanning left → right)')
    axes[1].set_yticks(range(N_QUBITS))
    plt.colorbar(im1, ax=axes[1])

    plt.tight_layout()
    return fig


# Pick 3 samples and show their qubit maps
torch.manual_seed(SEED)
sample_indices = np.random.choice(len(dataset), 3, replace=False)

figs = []
for idx in sample_indices:
    img, label = dataset[int(idx)]
    fig = plot_qubit_heatmap(q_model, img, label,
                              title_prefix=f'Sample {idx}')
    figs.append(fig)
    fig.savefig(os.path.join(PROJECT_PATH, f'qubit_heatmap_{idx}.png'),
                dpi=130, bbox_inches='tight')
    plt.show()

print('✅ Qubit heatmaps saved.')

## Cell 4 — Visualization 2: Qubit Signal Oscillations

In [ ]:
def plot_qubit_signals(model, img_tensor, label, save_path=None):
    """
    Line plot: each qubit's expectation value as the model
    scans across the image width (reading left→right).
    Different qubits pick up different spatial frequencies.
    """
    inp  = img_tensor.unsqueeze(0).to(device)
    post = model.get_post_quantum(inp).cpu().squeeze(0).numpy()  # [W, 8]

    palette = plt.cm.Set2(np.linspace(0, 1, N_QUBITS))

    fig, ax = plt.subplots(1, 1, figsize=(15, 4))
    for q in range(N_QUBITS):
        ax.plot(post[:, q] + q * 2.5, color=palette[q],
                linewidth=1.5, label=f'⟨Z{q}⟩')

    ax.set_title(f'Qubit Expectation Values  (⟨Z_i⟩)  —  Label: {label}',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Image Width Position (left → right)')
    ax.set_ylabel('Qubit State ⟨Z⟩  (offset for clarity)')
    ax.legend(ncol=4, loc='upper right', fontsize=9)
    ax.grid(alpha=0.25)
    plt.tight_layout()

    if save_path:
        fig.savefig(save_path, dpi=130, bbox_inches='tight')
    plt.show()
    return fig

# Show for one sample
idx = sample_indices[0]
img, label = dataset[int(idx)]
plot_qubit_signals(q_model, img, label,
    save_path=os.path.join(PROJECT_PATH, 'qubit_signals.png'))
print('✅ Saved qubit_signals.png')

## Cell 5 — Visualization 3: Visually Similar Characters Analysis
### Does quantum better distinguish '0'/'O', '1'/'I', '5'/'S'?

In [ ]:
def get_qubit_fingerprint(model, img_tensor):
    """Returns the mean qubit activation (8-dim vector) for an image."""
    inp  = img_tensor.unsqueeze(0).to(device)
    post = model.get_post_quantum(inp).cpu().squeeze(0).numpy()  # [W, 8]
    return post.mean(axis=0)  # [8] — averaged across width


# Find samples containing specific confusable characters
confusable_pairs = [('0', 'O'), ('1', 'I'), ('5', 'S'), ('8', 'B')]
char_fingerprints = {c: [] for pair in confusable_pairs for c in pair}

print('Collecting qubit fingerprints for confusable characters ...')
for i in range(min(500, len(dataset))):
    img, label = dataset[i]
    for char in char_fingerprints:
        if char in label and len(char_fingerprints[char]) < 20:
            fp = get_qubit_fingerprint(q_model, img)
            char_fingerprints[char].append(fp)

# Plot radar/bar for each pair
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Quantum Qubit Fingerprints: Confusable Character Pairs',
             fontsize=14, fontweight='bold')

colors_pair = [('#E91E63', '#FF9800'), ('#2196F3', '#4CAF50'),
               ('#9C27B0', '#00BCD4'), ('#F44336', '#8BC34A')]

for ax, (c1, c2), (col1, col2) in zip(axes.flatten(), confusable_pairs, colors_pair):
    fp1 = np.array(char_fingerprints[c1]) if char_fingerprints[c1] else np.zeros((1,8))
    fp2 = np.array(char_fingerprints[c2]) if char_fingerprints[c2] else np.zeros((1,8))
    m1  = fp1.mean(axis=0)
    m2  = fp2.mean(axis=0)
    s1  = fp1.std(axis=0)
    s2  = fp2.std(axis=0)

    x   = np.arange(N_QUBITS)
    w   = 0.35
    ax.bar(x - w/2, m1, w, color=col1, alpha=0.8, label=f"'{c1}'", yerr=s1)
    ax.bar(x + w/2, m2, w, color=col2, alpha=0.8, label=f"'{c2}'", yerr=s2)
    ax.set_xticks(x)
    ax.set_xticklabels([f'Q{i}' for i in range(N_QUBITS)])
    ax.set_title(f"'{c1}' vs '{c2}'  — Qubit Activation Difference")
    ax.set_ylabel('Mean ⟨Z⟩')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
out_path = os.path.join(PROJECT_PATH, 'char_confusion_qubits.png')
plt.savefig(out_path, dpi=130, bbox_inches='tight')
plt.show()
print('✅ Saved char_confusion_qubits.png')
print()
print('📝 Key insight for your report:')
print('   If the bars show DIFFERENT patterns for confusable pairs,')
print('   that means the quantum layer IS learning to distinguish them!')

## Cell 6 — Visualization 4: ZeroDCE Enhancement Quality Grid

In [ ]:
def apply_night(img_tensor, gamma=None):
    if gamma is None:
        gamma = np.random.uniform(2.0, 3.5)
    img = torch.pow(img_tensor, gamma)
    return torch.clamp(img + torch.randn_like(img) * 0.05, 0, 1)


# Show enhancement at different darkness levels
gammas = [1.5, 2.0, 2.5, 3.0, 3.5]
idx    = int(sample_indices[1])
img, label = dataset[idx]

fig, axes = plt.subplots(2, len(gammas)+1, figsize=(22, 5))
fig.suptitle(f'ZeroDCE Enhancement Quality vs Darkness Level — Label: {label}',
             fontsize=13, fontweight='bold')

# Original
axes[0, 0].imshow(img.permute(1,2,0).numpy())
axes[0, 0].set_title('Original', fontweight='bold')
axes[0, 0].axis('off')
axes[1, 0].axis('off')
axes[1, 0].text(0.5, 0.5, 'γ=1.0\n(clean)', ha='center', va='center',
                fontsize=10, bbox=dict(boxstyle='round', fc='#d4edda'))

for col, gamma in enumerate(gammas):
    dark = apply_night(img, gamma)
    inp  = dark.unsqueeze(0).to(device)
    with torch.no_grad():
        enhanced = q_model.enhancer(inp).cpu().squeeze(0)

    axes[0, col+1].imshow(np.clip(enhanced.permute(1,2,0).numpy(), 0, 1))
    axes[0, col+1].set_title(f'Enhanced  (γ={gamma})', fontsize=9)
    axes[0, col+1].axis('off')

    # Show dark input in row 2
    axes[1, col+1].imshow(np.clip(dark.permute(1,2,0).numpy(), 0, 1))
    axes[1, col+1].set_title(f'Dark Input  (γ={gamma})', fontsize=9,
                              color='gray')
    axes[1, col+1].axis('off')

plt.tight_layout()
out_path = os.path.join(PROJECT_PATH, 'zero_dce_quality.png')
plt.savefig(out_path, dpi=130, bbox_inches='tight')
plt.show()
print('✅ Saved zero_dce_quality.png')

## Cell 7 — Visualization 5: Training Curves (from saved history)

In [ ]:
HISTORY_DIR = os.path.join(PROJECT_PATH, 'history')

def load_history(name):
    path = os.path.join(HISTORY_DIR, f'{name}_history.json')
    if os.path.exists(path):
        with open(path) as f: return json.load(f)
    print(f'  ⚠️  No history for {name} yet — make sure Notebook 1 has run.')
    return None

q_hist = load_history('Quantum')
c_hist = load_history('Classical')

if q_hist and c_hist:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('Quantum ⚡ vs Classical 🔷 — Complete Training History',
                 fontsize=14, fontweight='bold')

    def plot_metric(ax, q_data, c_data, title, ylabel, invert=False):
        q_e = range(1, len(q_data)+1)
        c_e = range(1, len(c_data)+1)
        ax.plot(q_e, q_data, color='#7B2D8B', lw=2,   label='Quantum ⚡')
        ax.plot(c_e, c_data, color='#2196F3', lw=2,   label='Classical 🔷', ls='--')
        ax.fill_between(q_e, q_data, alpha=0.1, color='#7B2D8B')
        ax.fill_between(c_e, c_data, alpha=0.1, color='#2196F3')
        ax.set_title(title, fontweight='bold')
        ax.set_xlabel('Epoch')
        ax.set_ylabel(ylabel)
        ax.legend(); ax.grid(alpha=0.3)

    plot_metric(axes[0], q_hist['val_loss'], c_hist['val_loss'],
                'Validation Loss', 'CTC Loss')
    plot_metric(axes[1], q_hist['val_cer'],  c_hist['val_cer'],
                'Character Error Rate', 'CER ↓ (lower = better)')
    plot_metric(axes[2],
                [1 - v for v in q_hist['val_wer']],
                [1 - v for v in c_hist['val_wer']],
                'Plate Accuracy', 'Accuracy ↑ (higher = better)')

    plt.tight_layout()
    out_path = os.path.join(PROJECT_PATH, 'full_training_curves.png')
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Saved full_training_curves.png')
else:
    print('⚠️  Skipping training curves — run Notebook 1 to generate history files.')

## Cell 8 — Visualization 6: Architecture Diagram

In [ ]:
# Create a clean pipeline diagram for the paper/report
fig, ax = plt.subplots(1, 1, figsize=(16, 4))
ax.axis('off')

stages = [
    ('Input\nImage\n(Night)', '#555555'),
    ('ZeroDCE\nEnhancer', '#1565C0'),
    ('CNN Feature\nExtractor\n(→ 8ch)', '#1B5E20'),
    ('8-Qubit\nQuantum\nCircuit', '#6A1B9A'),
    ('Bi-LSTM\nDecoder', '#E65100'),
    ('CTC\nOutput\nText', '#BF360C'),
]

x_positions = np.linspace(0.05, 0.95, len(stages))

for x, (label, color) in zip(x_positions, stages):
    ax.add_patch(plt.FancyBboxPatch((x-0.065, 0.2), 0.12, 0.6,
                 boxstyle='round,pad=0.01',
                 facecolor=color, edgecolor='white', linewidth=2,
                 transform=ax.transAxes, clip_on=False))
    ax.text(x, 0.5, label, ha='center', va='center',
            color='white', fontsize=9.5, fontweight='bold',
            transform=ax.transAxes)

# Arrows between boxes
for i in range(len(x_positions)-1):
    ax.annotate('', xy=(x_positions[i+1]-0.065, 0.5),
                xytext=(x_positions[i]+0.065, 0.5),
                xycoords='axes fraction', textcoords='axes fraction',
                arrowprops=dict(arrowstyle='->', color='#333333', lw=2.5))

ax.set_title('HybridLPRNet_8Q — Quantum-Enhanced License Plate Recognition Pipeline',
             fontsize=13, fontweight='bold', pad=20)

out_path = os.path.join(PROJECT_PATH, 'architecture_diagram.png')
plt.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('✅ Saved architecture_diagram.png')
print()
print('✅ All visualizations complete!')
print(f'   All figures saved to: {PROJECT_PATH}')